# UNOCHA Bronze Layer - Exploratory Data Analysis (v2)

This notebook explores the bronze layer tables ingested from UN OCHA humanitarian datasets. The bronze layer contains tables in `workspace.default`:

- **18 CSV-based tables** — Allocations, COD Population (admin0-4), Contributions, FTS Funding, HPC HNO, Humanitarian Response Plans
- **7 INFORM Severity union tables** — Post-rebrand monthly crisis indices (all crises, country, impact, conditions, complexity, regional crises, trends) — each containing data from 67 monthly workbooks unioned together
- **~168 GCSI stage tables** — Pre-rebrand (2019-2020) GCSI-era Excel sheets (one table per file per sheet)
- **2 Manifest tables** — File and sheet inventory

**Goals:**
1. Inventory all bronze tables and understand their scale
2. Explore schemas and sample data for key datasets
3. Assess data quality (nulls, types, distributions)
4. Identify cross-dataset join keys for silver layer design
5. Understand the multi-header pattern in INFORM Severity sheets

## Exploratory Data Analysis

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd

TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"

# Get all bronze tables
all_tables_df = spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}")
bronze_tables = all_tables_df.filter(F.col("tableName").startswith(TARGET_PREFIX)).select("tableName").collect()
bronze_table_names = sorted([r["tableName"] for r in bronze_tables])
print(f"Total bronze tables: {len(bronze_table_names)}")

In [0]:
# Classify tables by type
csv_tables = []
inform_severity_tables = []
excel_stage_gcsi_tables = []
manifest_tables = []

INFORM_SEVERITY_PREFIX = f"{TARGET_PREFIX}_inform_severity_"
skip_suffixes = ("_manifest", "_sheet_manifest", "_excel_selected_sheets")

for tbl in bronze_table_names:
    if tbl.endswith(skip_suffixes):
        manifest_tables.append(tbl)
    elif tbl.startswith(INFORM_SEVERITY_PREFIX) and "_stage_" not in tbl:
        inform_severity_tables.append(tbl)
    elif "_stage_" in tbl and "gcsi" in tbl:
        excel_stage_gcsi_tables.append(tbl)
    elif "_stage_" in tbl:
        # Other stage tables (if any survived cleanup)
        excel_stage_gcsi_tables.append(tbl)
    else:
        csv_tables.append(tbl)

print(f"CSV bronze tables: {len(csv_tables)}")
print(f"INFORM Severity union tables: {len(inform_severity_tables)}")
print(f"GCSI/pre-rebrand stage tables: {len(excel_stage_gcsi_tables)}")
print(f"Manifest/metadata tables: {len(manifest_tables)}")

print(f"\n--- CSV Bronze Tables ---")
for tbl in csv_tables:
    dataset = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    print(f"  {dataset}")

print(f"\n--- INFORM Severity Union Tables (post-rebrand, 67 workbooks each) ---")
for tbl in inform_severity_tables:
    dataset = tbl.replace(f"{TARGET_PREFIX}_inform_severity_", "", 1)
    print(f"  {dataset}")

print(f"\n--- GCSI/Pre-rebrand Stage Tables (sample) ---")
gcsi_datasets = set()
for tbl in excel_stage_gcsi_tables:
    name = tbl.replace(f"{TARGET_PREFIX}_stage_", "", 1)
    # Extract month grouping
    for suffix in ["core_indicators", "crisis_indicator_data", "country_indicator_data", 
                   "complexity", "conditions_of_affected_people", "impact_of_crisis", 
                   "gcsi", "regional_crises", "reliability"]:
        if name.endswith(suffix):
            gcsi_datasets.add(suffix)
            break
print(f"  Sheet categories: {sorted(gcsi_datasets)}")
print(f"  Total stage tables: {len(excel_stage_gcsi_tables)}")

print(f"\n--- Manifest tables ---")
for tbl in manifest_tables:
    print(f"  {tbl}")

In [0]:
# Explore manifest/metadata tables
for tbl in manifest_tables:
    full_name = f"{TARGET_SCHEMA}.{tbl}"
    dataset = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    print(f"\n{'='*80}")
    print(f"MANIFEST TABLE: {dataset}")
    print(f"{'='*80}")
    try:
        schema_df = spark.sql(f"DESCRIBE TABLE {full_name}")
        display(schema_df)
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0]["cnt"]
        print(f"Rows: {row_count:,}")
        sample_df = spark.sql(f"SELECT * FROM {full_name} LIMIT 5")
        display(sample_df)
    except Exception as e:
        print(f"  ERROR: {str(e)[:200]}")

In [0]:
# Explore key CSV datasets
key_csv_tables = [
    "njyoti_bronze_unocha_allocations",
    "njyoti_bronze_unocha_cod_population_admin0",
    "njyoti_bronze_unocha_fts_incoming_funding_global",
    "njyoti_bronze_unocha_hpc_hno_2025"
]

for tbl in key_csv_tables:
    full_name = f"{TARGET_SCHEMA}.{tbl}"
    dataset = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    print(f"\n{'='*80}")
    print(f"TABLE: {dataset}")
    print(f"{'='*80}")
    
    try:
        # Use DESCRIBE to get column names safely
        schema_df = spark.sql(f"DESCRIBE TABLE {full_name}")
        cols_info = schema_df.collect()
        # Filter to actual data columns (exclude partition info rows)
        col_names = [row["col_name"] for row in cols_info if row["col_name"] and row["data_type"] and not row["col_name"].startswith("#")]
        print(f"\nSchema ({len(col_names)} columns):")
        for cn in col_names[:20]:
            dtype = next((row["data_type"] for row in cols_info if row["col_name"] == cn), "")
            print(f"  {cn:40s} {dtype:15s}")
        if len(col_names) > 20:
            print(f"  ... and {len(col_names) - 20} more columns")
        
        # Exclude _sheet_name which exists in schema but not in data files (Photon bug)
        safe_cols = [c for c in col_names if c != "_sheet_name"]
        col_list = ", ".join([f"`{c}`" for c in safe_cols])
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0]["cnt"]
        print(f"\nRows: {row_count:,} | Columns: {len(col_names)}")
        print(f"\nSample (5 rows):")
        sample_df = spark.sql(f"SELECT {col_list} FROM {full_name} LIMIT 5")
        display(sample_df)
    except Exception as e:
        print(f"  ERROR: {str(e)[:200]}")
        print(f"  Trying fallback with DESCRIBE only...")
        display(spark.sql(f"DESCRIBE TABLE {full_name}"))

In [0]:
# Explore the 7 new INFORM Severity union tables
# These contain data from 67 monthly workbooks (Oct 2020 - Apr 2026) unioned together
# Key insight: 'regional_crises' has clean headers; the other 6 have multi-header issues

INFORM_TABLES = [
    ("regional_crises", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_regional_crises"),
    ("all_crises", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_all_crises"),
    ("country", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_country"),
    ("impact_of_the_crisis", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_impact_of_the_crisis"),
    ("conditions_of_people_affected", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_conditions_of_people_affected"),
    ("complexity_of_the_crisis", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_complexity_of_the_crisis"),
    ("trends", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_trends"),
]

print("=" * 80)
print("INFORM SEVERITY UNION TABLES - OVERVIEW")
print("=" * 80)

for label, full_name in INFORM_TABLES:
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0]["cnt"]
        cols_info = spark.sql(f"DESCRIBE TABLE {full_name}").collect()
        col_names = [row["col_name"] for row in cols_info if row["col_name"] and not row["col_name"].startswith("#")]
        source_files = spark.sql(f"SELECT COUNT(DISTINCT _source_file_name) as cnt FROM {full_name}").collect()[0]["cnt"]
        print(f"\n  {label:40s} | {row_count:>8,} rows | {len(col_names):>3} cols | {source_files} source files")
    except Exception as e:
        print(f"\n  {label:40s} | ERROR: {str(e)[:80]}")

# --- Explore Regional Crises (the one with clean headers) ---
print(f"\n\n{'='*80}")
print("REGIONAL CRISES - Clean headers, immediately queryable")
print(f"{'='*80}")

rc_table = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_regional_crises"
try:
    schema_df = spark.sql(f"DESCRIBE TABLE {rc_table}")
    cols_info = schema_df.collect()
    col_names = [row["col_name"] for row in cols_info if row["col_name"] and not row["col_name"].startswith("#")]
    # Exclude metadata columns for display
    data_cols = [c for c in col_names if not c.startswith("_")]
    meta_cols = [c for c in col_names if c.startswith("_")]
    
    print(f"\nData columns ({len(data_cols)}):")
    for cn in data_cols:
        dtype = next((row["data_type"] for row in cols_info if row["col_name"] == cn), "")
        print(f"  {cn:50s} {dtype}")
    print(f"\nMetadata columns ({len(meta_cols)}): {meta_cols}")
    
    print(f"\nSample data (5 rows):")
    safe_cols = ", ".join([f"`{c}`" for c in data_cols[:10]] + ["`_source_file_name`"])
    display(spark.sql(f"SELECT {safe_cols} FROM {rc_table} LIMIT 5"))
except Exception as e:
    print(f"ERROR: {str(e)[:200]}")

# --- Show the multi-header pattern in other tables ---
print(f"\n\n{'='*80}")
print("MULTI-HEADER PATTERN - Example from 'all_crises' table")
print("(First few rows contain header metadata, not data)")
print(f"{'='*80}")

ac_table = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_all_crises"
try:
    cols_info = spark.sql(f"DESCRIBE TABLE {ac_table}").collect()
    col_names = [row["col_name"] for row in cols_info if row["col_name"] and not row["col_name"].startswith("#") and not row["col_name"].startswith("_")]
    print(f"\nColumn names (first 15 of {len(col_names)}):")
    for c in col_names[:15]:
        print(f"  {c}")
    print(f"\nNote: columns named 'unnamed_*' or containing release dates indicate")
    print(f"the multi-header issue. Real data starts at row 2-3 in most sheets.")
    print(f"This will be resolved in the silver layer.")
    print(f"\nFirst 5 rows (showing header contamination):")
    safe_cols = ", ".join([f"`{c}`" for c in col_names[:8]])
    display(spark.sql(f"SELECT {safe_cols} FROM {ac_table} LIMIT 5"))
except Exception as e:
    print(f"ERROR: {str(e)[:200]}")

In [0]:
# Data quality assessment on key tables
# Using DESCRIBE to get safe column list and avoiding _sheet_name issues
#
# TABLE SELECTION RATIONALE:
# We pick one representative table from each major category identified in cell 4:
#   - allocations: humanitarian funding allocations (CSV)
#   - cod_population_admin0: population reference data (CSV)
#   - fts_incoming_funding_global: financial tracking (CSV, largest dataset)
#   - hpc_hno_2025: humanitarian needs overview (CSV)
#   - humanitarian_response_plans: response plan metadata (CSV)
#   - inform_severity_regional_crises: the only INFORM Severity table with clean
#     headers (the other 6 have multi-header contamination from Excel, which would
#     skew null-percentage calculations)
#
# EXCLUDED from this assessment:
#   - Other INFORM Severity tables (6): multi-header rows would inflate null %
#     misleadingly — these need silver-layer cleanup first
#   - GCSI stage tables (~168): too numerous for individual assessment; same
#     multi-header pattern applies
#   - Manifest tables (2): metadata-only, not analytical data
#
# Dataset categories are derived from the classification logic in Cell 4 which
# groups all bronze tables by naming convention (prefix/suffix patterns).

quality_tables = [
    "njyoti_bronze_unocha_allocations",
    "njyoti_bronze_unocha_cod_population_admin0",
    "njyoti_bronze_unocha_fts_incoming_funding_global",
    "njyoti_bronze_unocha_hpc_hno_2025",
    "njyoti_bronze_unocha_humanitarian_response_plans",
    "njyoti_bronze_unocha_inform_severity_regional_crises",
]

quality_results = []

for tbl in quality_tables:
    full_name = f"{TARGET_SCHEMA}.{tbl}"
    dataset = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0]["cnt"]
        if row_count == 0:
            print(f"\n  {dataset}: EMPTY TABLE")
            continue
        
        # Get column names via DESCRIBE
        cols_info = spark.sql(f"DESCRIBE TABLE {full_name}").collect()
        col_names = [row["col_name"] for row in cols_info 
                     if row["col_name"] and not row["col_name"].startswith("#")
                     and row["data_type"]]  # only real columns with types
        
        # Exclude void-type columns (_sheet_name is sometimes void)
        safe_cols = [row["col_name"] for row in cols_info 
                     if row["col_name"] and not row["col_name"].startswith("#")
                     and row["data_type"] and row["data_type"] != "void"]
        
        # Calculate null percentages using SQL
        null_exprs = ", ".join([
            f"ROUND(SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) / {row_count} * 100, 1) AS `null_pct_{c}`" 
            for c in safe_cols
        ])
        null_pcts_row = spark.sql(f"SELECT {null_exprs} FROM {full_name}").collect()[0]
        
        print(f"\n{'='*80}")
        print(f"DATA QUALITY: {dataset} ({row_count:,} rows, {len(safe_cols)} cols)")
        print(f"{'='*80}")
        
        # Show columns with high null rates
        high_nulls = []
        low_nulls = []
        for c in safe_cols:
            pct = null_pcts_row[f"null_pct_{c}"]
            if pct is not None:
                if float(pct) > 10:
                    high_nulls.append((c, float(pct)))
                else:
                    low_nulls.append((c, float(pct)))
        
        print(f"\nColumns with >10% nulls ({len(high_nulls)}/{len(safe_cols)}):")
        for col, pct in sorted(high_nulls, key=lambda x: -x[1])[:15]:
            bar = '\u2588' * int(pct // 5)
            print(f"  {col:40s} {pct:5.1f}% {bar}")
        if len(high_nulls) > 15:
            print(f"  ... and {len(high_nulls) - 15} more")
        
        print(f"\nClean columns (<=10% nulls): {len(low_nulls)}/{len(safe_cols)}")
        
        # Data type analysis
        type_counts = {}
        for row in cols_info:
            if row["col_name"] in safe_cols:
                t = row["data_type"]
                type_counts[t] = type_counts.get(t, 0) + 1
        print(f"Column types: {type_counts}")
        
        quality_results.append({
            "dataset": dataset,
            "rows": row_count,
            "columns": len(safe_cols),
            "high_null_cols": len(high_nulls),
            "clean_cols": len(low_nulls)
        })
    except Exception as e:
        print(f"\nERROR processing {dataset}: {str(e)[:200]}")

# Summary
print(f"\n{'='*80}")
print("QUALITY SUMMARY")
print(f"{'='*80}")
for r in quality_results:
    print(f"  {r['dataset']:45s} {r['rows']:>10,} rows | {r['clean_cols']}/{r['columns']} clean cols")

In [0]:
# Cross-dataset join key analysis
# Identify common columns that can link datasets together
#
# JOIN KEY CANDIDATE SELECTION RATIONALE:
# These candidates were identified by inspecting the actual column names from
# Cell 6's schema exploration of key CSV and INFORM tables:
#   - "iso3", "country", "country_code", "iso3_code": Geographic identifiers
#     appearing across COD Population (iso3, country), FTS Funding (destlocations
#     contains ISO3 codes), HPC HNO (country_iso3), INFORM Severity (iso3/country)
#   - "crisis_id": Key identifier in INFORM Severity tables for linking crisis
#     assessments across sub-indices
#   - "admin0", "admin1", "admin2": Administrative boundary levels present in
#     COD Population (adm1_pcode, adm2_pcode) and HPC HNO (admin_1_pcode, etc.)
#   - "year": Temporal dimension present in allocations, COD population
#     (reference_year), FTS funding (budgetyear), and derivable from INFORM
#     source file names
#   - "plan_id", "sector": Humanitarian plan identifiers linking Allocations,
#     HPC HNO, and Humanitarian Response Plans (destplanid in FTS, sector in HNO)
#
# The search uses substring matching ("key in col_name") to catch variants like
# "country_iso3", "destplanid", "admin_1_pcode" etc. without needing exact names.

join_key_candidates = ["iso3", "country", "crisis_id", "iso3_code", "country_code", 
                       "admin0", "admin1", "admin2", "year", "plan_id", "sector"]

analysis_tables = [
    ("allocations", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_allocations"),
    ("cod_population_admin0", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_cod_population_admin0"),
    ("cod_population_admin1", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_cod_population_admin1"),
    ("fts_incoming_funding", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_fts_incoming_funding_global"),
    ("hpc_hno_2025", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_hpc_hno_2025"),
    ("humanitarian_response_plans", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_humanitarian_response_plans"),
    ("inform_severity_regional_crises", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_regional_crises"),
    ("inform_severity_all_crises", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_all_crises"),
    ("inform_severity_country", f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_country"),
]

print("CROSS-DATASET JOIN KEY ANALYSIS")
print(f"{'='*80}")
print(f"\nLooking for columns matching: {join_key_candidates}")
print()

# Build a matrix of which tables have which join keys
join_matrix = {}
for label, full_name in analysis_tables:
    try:
        cols_info = spark.sql(f"DESCRIBE TABLE {full_name}").collect()
        col_names = [row["col_name"].lower() for row in cols_info if row["col_name"] and not row["col_name"].startswith("#")]
        col_names_orig = [row["col_name"] for row in cols_info if row["col_name"] and not row["col_name"].startswith("#")]
        
        found_keys = []
        for key in join_key_candidates:
            matches = [col_names_orig[i] for i, low in enumerate(col_names) if key in low]
            if matches:
                found_keys.extend(matches)
        join_matrix[label] = found_keys
    except Exception as e:
        join_matrix[label] = [f"ERROR: {str(e)[:50]}"]

# Display matrix
print(f"{'Dataset':<35s} | Join Key Columns Found")
print(f"{'-'*35}-+-{'-'*60}")
for label, keys in join_matrix.items():
    keys_str = ", ".join(keys[:10])
    if len(keys) > 10:
        keys_str += f" ... (+{len(keys)-10} more)"
    print(f"{label:<35s} | {keys_str}")

# Identify best linkage paths
print(f"\n\n{'='*80}")
print("RECOMMENDED JOIN PATHS FOR SILVER LAYER")
print(f"{'='*80}")
print("""
1. Country-level joins:
   - COD Population (admin0) <-> FTS Funding <-> HPC HNO <-> INFORM Severity
   - Key: ISO3 code / country name
   - Note: INFORM Severity tables with multi-header issues need silver processing first

2. Crisis-level joins:
   - INFORM Severity (regional_crises has clean headers) <-> FTS Funding
   - Key: crisis_id / crisis name

3. Plan/Sector joins:
   - Allocations <-> HPC HNO <-> Humanitarian Response Plans
   - Key: plan_id / sector

4. Temporal joins:
   - All datasets can be linked by year for trend analysis
   - INFORM Severity tables have _source_file_name for monthly provenance

5. INFORM Severity internal joins:
   - Regional Crises <-> All Crises <-> Country (via crisis/country identifiers)
   - Requires silver layer to resolve multi-header structure first
""")

In [0]:
# Temporal coverage analysis across INFORM Severity tables
# Since each table unions 67 monthly workbooks, check time span and completeness

import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

print("TEMPORAL COVERAGE - INFORM Severity Tables")
print("=" * 80)
print("\nEach row's source month is embedded in _source_file_name.")
print("Checking how many distinct source files contributed to each table:\n")

for label, full_name in INFORM_TABLES:
    try:
        result = spark.sql(f"""
            SELECT 
                COUNT(*) as total_rows,
                COUNT(DISTINCT _source_file_name) as source_files,
                MIN(_source_file_name) as earliest_file,
                MAX(_source_file_name) as latest_file
            FROM {full_name}
        """).collect()[0]
        print(f"  {label:40s} | {result['total_rows']:>7,} rows | {result['source_files']:>3} files | {result['earliest_file'][:30]}...{result['latest_file'][-30:]}")
    except Exception as e:
        print(f"  {label:40s} | ERROR: {str(e)[:60]}")

# Show distribution of rows per source file for regional_crises (clean data)
print(f"\n\n{'='*80}")
print("ROWS PER SOURCE FILE - Regional Crises (clean headers)")
print(f"{'='*80}")

rc_table = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_regional_crises"
rows_per_file = spark.sql(f"""
    SELECT _source_file_name, COUNT(*) as row_count
    FROM {rc_table}
    GROUP BY _source_file_name
    ORDER BY _source_file_name
""").toPandas()

# Extract month-year from file names for readable x-axis
MONTH_MAP = {
    'january': 1, 'february': 2, 'march': 3, 'april': 4,
    'may': 5, 'june': 6, 'july': 7, 'august': 8,
    'september': 9, 'october': 10, 'november': 11, 'december': 12
}

def parse_month_year(filename):
    """Extract a datetime from INFORM Severity file names."""
    fn = filename.lower()
    for month_name, month_num in MONTH_MAP.items():
        if month_name in fn:
            # Find the year (4 digits after the month or in the prefix)
            year_match = re.search(rf'{month_name}[- _](\d{{4}})', fn)
            if year_match:
                return datetime(int(year_match.group(1)), month_num, 1)
            # Try prefix date like 202201-inform-severity-...
            prefix_match = re.match(r'(\d{4})', fn)
            if prefix_match:
                return datetime(int(prefix_match.group(1)), month_num, 1)
    return None

rows_per_file['month_date'] = rows_per_file['_source_file_name'].apply(parse_month_year)
rows_per_file = rows_per_file.dropna(subset=['month_date']).sort_values('month_date')

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(rows_per_file['month_date'], rows_per_file['row_count'], width=25, color='steelblue', alpha=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Regional Crises')
ax.set_title('INFORM Severity: Regional Crises Count Over Time (53 of 67 months)')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.set_ylim(0, max(rows_per_file['row_count']) + 2)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nNote: regional_crises has 53 source files vs. 67 expected.")
print(f"Latest file: {rows_per_file['_source_file_name'].iloc[-1]}")
print(f"Crises tracked: {rows_per_file['row_count'].min()}-{rows_per_file['row_count'].max()} per month")

In [0]:
%sql
SELECT * FROM workspace.default.njyoti_bronze_unocha_inform_severity_country LIMIT 5